# CNC tool-wear detection — Anomaly detection model design

Forgewright Manufacturing, one production shift across a CNC cell. Three exports:
per-second **power**, ~2 Hz **vibration**, and the **MES production log** of jobs.
Goal: per-job mean power and peak vibration, and a ranked list of jobs that show
tool wear.

**The physics that drives every decision below.** A dull or chipped tool chatters:
it vibrates more than a healthy tool would *for the same cutting work*. Cutting work
shows up as power draw. So wear is **high vibration relative to power** — not high
vibration, and not high power. A heavy titanium cut is legitimately loud and must not
be flagged. This rules out any absolute vibration threshold from the start.

The EDA established the physics and the shape of the data. This notebook tests the
candidate architectures and picks the pair that goes into `analyse.py`.

**Architecture.** Two stages, combined with a logical AND.

* **Primary** -- a regression of vibration on cutting power plus job context. Flag
  readings whose residual exceeds **6 robust sigma**. This is the physics: how much more
  did the spindle vibrate than a healthy tool would for this much cutting work?
  Candidates: **ridge**, **Huber**, **RANSAC**.
* **Secondary** -- an unsupervised multivariate outlier detector in sensor space.
  Candidates: **Isolation Forest**, **Mahalanobis distance**, **Local Outlier Factor**.

A reading counts as anomalous only if both stages fire. A job is flagged when a
sustained share of its readings are anomalous.

**How we judge, with no answer key.** The EDA found two *independent* signals that agree
perfectly on the same 11 jobs: vibration-per-kW relative to a robust part-type baseline
(bimodal, with an empty gap) and the crest factor (perfectly separated). We reconstruct
that set here and use it as a **reference set** for comparing models. It is not ground
truth and the pipeline never uses it -- but a detector that disagrees with two
independent physical signals needs to justify itself.

**What the profile turns up**

| Issue | Evidence | Handling |
|---|---|---|
| Sensor clocks vs MES clock | sensors run 13:00–21:00 UTC, MES 08:03–13:42 naive — they barely overlap | estimate the offset from the data (section 3) |
| Sentinel power values | a handful of `9999.0` rows, vs a 0.999 quantile near 19 kW | drop via a loose IQR fence |
| Negative power | a few `-3.0` rows | drop: physically impossible |
| Duplicate readings | a few rows sharing (machine, timestamp), identical values | drop, keep first |
| Jobs with `end < start` | 3 jobs, small negative durations | transposition, not a real event → swap |
| Missing `quantity` | 3 jobs | impute with the part-type median, and record that we did |
| Vibration dropouts | two multi-minute gaps | tolerate; coverage is reported per job |

The fences are deliberately loose. A genuinely heavy titanium cut must survive; only
impossible values should go.

In [1]:
import warnings
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DATA_DIR = "../data"
OUT_DIR = "../output"
# --- Tunables. Every decision the analysis makes, in one place. -------------- #
# Nothing here names a machine, job or date: the fleet is discovered from the data.
POWER_MIN_KW = 0.0            # negative draw is physically impossible
POWER_ABS_MAX_KW = 500.0      # hard ceiling for this class of machine
IQR_FENCE = 50.0              # very loose fence, to catch 9999-style sentinels only
CUT_FLOOR_KW = 0.5            # below this the machine is not really cutting
SIGMA_MULTIPLE = 6.0          # the 6-sigma rule for the primary detector
TRIM_SIGMA = 3.0              # trimming point for the healthy-scale estimator
CONTAMINATION = 0.15          # secondary-detector prior; see section 7
MIN_ANOM_READINGS = 5         # a job needs sustained evidence, not one spike
MIN_ANOM_FRACTION = 0.05
MIN_READINGS_VERDICT = 20     # too little data -> no verdict rather than a guess
SCORE_QUANTILE = 0.95         # wear_score = 95th percentile of residual sigma
RANDOM_STATE = 7

## 1. Load the raw exports

Timestamps first, because they turn out to be the main hazard. The sensor files carry
an explicit UTC offset (`...Z`); the MES log is naive local time. Everything is parsed
to naive-UTC here and any residual offset is *estimated from the data* in section 3 —
never assumed from a timezone rule.

In [2]:
def parse_utc(series):
    '''Parse mixed ISO timestamps to tz-naive UTC.'''
    return pd.to_datetime(series, format="mixed", utc=True, errors="coerce").dt.tz_localize(None)


def load_power(path):
    df = pd.read_csv(path)[["timestamp", "machine_id", "power_kw"]].copy()
    df["timestamp"] = parse_utc(df["timestamp"])
    df["machine_id"] = df["machine_id"].astype("string").str.strip()
    df["power_kw"] = pd.to_numeric(df["power_kw"], errors="coerce")
    return df


def load_vibration(path):
    cols = ["timestamp", "machine_id", "vibration_rms_g", "vibration_peak_g"]
    df = pd.read_csv(path)[cols].copy()
    df["timestamp"] = parse_utc(df["timestamp"])
    df["machine_id"] = df["machine_id"].astype("string").str.strip()
    for c in ("vibration_rms_g", "vibration_peak_g"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def load_production_log(path):
    df = pd.read_csv(path)
    df["machine_id"] = df["machine_id"].astype("string").str.strip()
    for c in ("start_time", "end_time"):
        df[c] = parse_utc(df[c])
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    return df


power_raw = load_power(os.path.join(DATA_DIR, "power.csv"))
vib_raw = load_vibration(os.path.join(DATA_DIR, "vibration.csv"))
log_raw = load_production_log(os.path.join(DATA_DIR, "production_log.csv"))

## 2. Define the cleaning functions

In [3]:
def robust_upper_bound(values, iqr_multiple=IQR_FENCE):
    '''Loose upper plausibility fence, wide enough to keep real heavy cuts.'''
    v = values[np.isfinite(values)]
    if v.size == 0:
        return np.inf
    q1, q3 = np.percentile(v, [25, 75])
    return float(q3 + iqr_multiple * max(q3 - q1, 1e-9))


def clean_power(power):
    df = power.dropna(subset=["timestamp", "machine_id", "power_kw"]).copy()
    n0 = len(df)
    df = df.drop_duplicates(subset=["machine_id", "timestamp"], keep="first")
    n_dup = n0 - len(df)

    neg = df["power_kw"] < POWER_MIN_KW
    bound = min(POWER_ABS_MAX_KW, robust_upper_bound(df.loc[~neg, "power_kw"].to_numpy(float)))
    sentinel = df["power_kw"] > bound
    report = {"rows_in": len(power), "duplicates": n_dup, "negative": int(neg.sum()),
              "sentinel": int(sentinel.sum()), "sentinel_bound_kw": round(bound, 2),
              "sentinel_values": sorted(df.loc[sentinel, "power_kw"].unique().tolist())}
    df = df[~neg & ~sentinel].sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
    report["rows_out"] = len(df)
    return df, report


def clean_vibration(vibration):
    cols = ["timestamp", "machine_id", "vibration_rms_g", "vibration_peak_g"]
    df = vibration.dropna(subset=cols).copy()
    n0 = len(df)
    df = df.drop_duplicates(subset=["machine_id", "timestamp"], keep="first")
    n_dup = n0 - len(df)

    bad = ((df["vibration_rms_g"] < 0) | (df["vibration_peak_g"] < 0)
           | (df["vibration_peak_g"] < df["vibration_rms_g"]))
    report = {"rows_in": len(vibration), "duplicates": n_dup, "invalid": int(bad.sum())}
    df = df[~bad]
    for col in ("vibration_rms_g", "vibration_peak_g"):
        bound = robust_upper_bound(df[col].to_numpy(float))
        report[f"sentinel_{col}"] = int((df[col] > bound).sum())
        df = df[df[col] <= bound]
    df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
    report["rows_out"] = len(df)
    return df, report


def clean_production_log(log):
    df = log.dropna(subset=["start_time", "end_time"]).drop_duplicates(subset=["job_id"]).copy()
    df["notes"] = ""

    reversed_mask = df["end_time"] < df["start_time"]
    if reversed_mask.any():
        s = df.loc[reversed_mask, "start_time"].copy()
        df.loc[reversed_mask, "start_time"] = df.loc[reversed_mask, "end_time"]
        df.loc[reversed_mask, "end_time"] = s
        df.loc[reversed_mask, "notes"] = "start/end transposed in MES; swapped"

    df["duration_s"] = (df["end_time"] - df["start_time"]).dt.total_seconds()
    df = df[df["duration_s"] >= 1.0].copy()

    missing_qty = df["quantity"].isna()
    df["quantity_imputed"] = missing_qty
    if missing_qty.any():
        by_part = df.groupby("part_type")["quantity"].transform("median")
        df["quantity"] = df["quantity"].fillna(by_part).fillna(df["quantity"].median())
        df.loc[missing_qty, "notes"] = (df.loc[missing_qty, "notes"] + "; quantity imputed").str.strip("; ")

    df = df.sort_values(["machine_id", "start_time"]).reset_index(drop=True)
    report = {"rows_in": len(log), "rows_out": len(df),
              "reversed": int(reversed_mask.sum()), "quantity_imputed": int(missing_qty.sum())}

    # Overlapping jobs on one machine would make the reading->job join ambiguous.
    overlaps = []
    for _, g in df.groupby("machine_id"):
        prev_end, prev_id = None, None
        for row in g.itertuples():
            if prev_end is not None and row.start_time < prev_end:
                overlaps.append(f"{prev_id}->{row.job_id}")
            prev_end, prev_id = row.end_time, row.job_id
    report["overlapping_job_pairs"] = overlaps
    return df, report


power, power_report = clean_power(power_raw)
vib, vib_report = clean_vibration(vib_raw)
log, log_report = clean_production_log(log_raw)

for name, rep in [("power", power_report), ("vibration", vib_report), ("production_log", log_report)]:
    print(f"{name:<15} {rep}")

power           {'rows_in': 86405, 'duplicates': 5, 'negative': 3, 'sentinel': 4, 'sentinel_bound_kw': 421.29, 'sentinel_values': [9999.0], 'rows_out': 86377}
vibration       {'rows_in': 168969, 'duplicates': 8, 'invalid': 0, 'sentinel_vibration_rms_g': 0, 'sentinel_vibration_peak_g': 0, 'rows_out': 168961}
production_log  {'rows_in': 94, 'rows_out': 94, 'reversed': 3, 'quantity_imputed': 3, 'overlapping_job_pairs': []}


## 3. Recovering the clock offset from the data

The hint in the brief is that sensor timestamps should fall between a job's `start_time`
and `end_time`. As loaded, they mostly don't. Rather than assume "UTC+5", **estimate**
the offset with a physical objective: cutting draws power and idling does not, so the
correct offset is the one that maximises *in-job minus out-of-job* mean power. Coarse
hourly sweep, then a fine one-minute sweep.

Why this matters for generalisation: a held-out shift from other machines may carry a
different offset, and this discovers it instead of hardcoding one.

In [4]:
def in_job_mask(timestamps, jobs, shift_s=0.0):
    '''Boolean mask: does each timestamp fall inside any of this machine's job windows?'''
    shift = pd.to_timedelta(shift_s, unit="s")
    inside = np.zeros(len(timestamps), dtype=bool)
    starts = (jobs["start_time"] + shift).to_numpy()
    ends = (jobs["end_time"] + shift).to_numpy()
    for s, e in zip(starts, ends):
        inside |= (timestamps >= s) & (timestamps <= e)
    return inside


def coverage_separation(power_df, jobs_df, offset_s):
    '''Mean in-job minus out-of-job power lift, averaged over machines.'''
    seps = []
    for machine, g in power_df.groupby("machine_id", observed=True):
        jobs = jobs_df[jobs_df["machine_id"] == machine]
        if jobs.empty:
            continue
        inside = in_job_mask(g["timestamp"].to_numpy(), jobs, offset_s)
        if inside.all() or not inside.any():
            continue
        p = g["power_kw"].to_numpy(float)
        seps.append(float(p[inside].mean() - p[~inside].mean()))
    return float(np.mean(seps)) if seps else -np.inf


def estimate_clock_offset(power_df, jobs_df, coarse_range_h=14, fine_window_s=3600, fine_step_s=60):
    coarse = {h * 3600: coverage_separation(power_df, jobs_df, h * 3600)
              for h in range(-coarse_range_h, coarse_range_h + 1)}
    best_coarse = max(coarse, key=coarse.get)
    fine = {best_coarse + s: coverage_separation(power_df, jobs_df, best_coarse + s)
            for s in range(-fine_window_s, fine_window_s + 1, fine_step_s)}
    best = max(fine, key=fine.get)
    return best, coarse, fine


offset_s, coarse_scores, fine_scores = estimate_clock_offset(power, log)

print("hourly sweep (power lift in kW):")
for off, score in sorted(coarse_scores.items()):
    if np.isfinite(score):
        marker = "  <== best" if off == max(coarse_scores, key=coarse_scores.get) else ""
        print(f"  {off / 3600:+.0f} h -> {score:+.3f}{marker}")
print(f"\nchosen offset: {offset_s / 3600:+.2f} h ({offset_s:+.0f} s)")
print(f"power lift at 0 h: {coarse_scores.get(0, float('nan')):+.3f} kW -> "
      f"at chosen offset: {fine_scores[offset_s]:+.3f} kW")

# Do the machines agree independently? If they didn't, a single global shift would be wrong.
per_machine = {}
for machine in sorted(power["machine_id"].unique()):
    pm = power[power["machine_id"] == machine]
    lm = log[log["machine_id"] == machine]
    scores = {h * 3600: coverage_separation(pm, lm, h * 3600) for h in range(-14, 15)}
    per_machine[machine] = max(scores, key=scores.get) / 3600
print(f"per-machine best offset (h): {per_machine}")
print(f"all machines agree: {len(set(per_machine.values())) == 1}")

log["start_time_aligned"] = log["start_time"] + pd.to_timedelta(offset_s, unit="s")
log["end_time_aligned"] = log["end_time"] + pd.to_timedelta(offset_s, unit="s")

hourly sweep (power lift in kW):
  +0 h -> -1.348
  +1 h -> +0.498
  +2 h -> +2.407
  +3 h -> +2.979
  +4 h -> +3.888
  +5 h -> +8.377  <== best
  +6 h -> +2.249
  +7 h -> -0.109
  +8 h -> -2.060
  +9 h -> -4.381
  +10 h -> -4.927
  +11 h -> -4.985
  +12 h -> -4.457

chosen offset: +5.00 h (+18000 s)
power lift at 0 h: -1.348 kW -> at chosen offset: +8.377 kW
per-machine best offset (h): {'CNC-07': 5.0, 'CNC-09': 5.0, 'CNC-11': 5.0}
all machines agree: True


## 4. Join readings to jobs, and measure idle draw

Three steps:

1. **Label** each reading with the job running on its machine at that instant — a
   per-machine `searchsorted` over job starts, no assumptions about naming or ordering.
2. **Pair** each vibration reading with the nearest power reading. Vibration logs ~2 Hz
   and power ~1 Hz, so `merge_asof` with a 1 s tolerance matches them at native
   resolution instead of resampling onto an arbitrary grid — which matters because the
   deliverable needs the *peak* vibration.
3. **Idle baseline** per machine, measured on readings *outside* every job window. The
   brief defines mean job power as net of this idle draw.

Readings outside all job windows are exactly what makes the idle estimate possible, so
they're kept rather than discarded.

In [5]:
def assign_jobs(readings, jobs_df):
    '''Label each reading with its job_id (NA when the machine is between jobs).'''
    out = readings.copy()
    out["job_id"] = pd.Series(pd.NA, index=out.index, dtype="string")
    for machine, jobs in jobs_df.groupby("machine_id", observed=True):
        mask = (out["machine_id"] == machine).to_numpy()
        if not mask.any():
            continue
        jobs = jobs.sort_values("start_time_aligned")
        starts = jobs["start_time_aligned"].to_numpy()
        ends = jobs["end_time_aligned"].to_numpy()
        ids = jobs["job_id"].to_numpy()

        t = out.loc[mask, "timestamp"].to_numpy()
        pos = np.searchsorted(starts, t, side="right") - 1
        valid = (pos >= 0) & (t <= ends[np.clip(pos, 0, len(ends) - 1)])
        labels = np.full(len(t), None, dtype=object)
        labels[valid] = ids[pos[valid]]
        out.iloc[np.flatnonzero(mask), out.columns.get_loc("job_id")] = pd.array(labels, dtype="string")
    return out


def pair_vibration_with_power(vibration, power_df, tolerance_s=1.0):
    '''Nearest-in-time power reading for each vibration reading, per machine.'''
    return pd.merge_asof(
        vibration.sort_values("timestamp"),
        power_df[["timestamp", "machine_id", "power_kw"]].sort_values("timestamp"),
        on="timestamp", by="machine_id", direction="nearest",
        tolerance=pd.Timedelta(seconds=tolerance_s),
    )


def estimate_idle_baseline(power_with_jobs):
    '''Minimum power draw per machine while no job is running.'''
    idle = power_with_jobs[power_with_jobs["job_id"].isna()]
    baseline = idle.groupby("machine_id", observed=True)["power_kw"].min()
    machines = pd.Index(sorted(power_with_jobs["machine_id"].unique()), name="machine_id")
    fallback = float(idle["power_kw"].min()) if len(idle) else 0.0
    return baseline.reindex(machines).fillna(fallback)


power = assign_jobs(power, log)
vib = assign_jobs(vib, log)
idle_baseline = estimate_idle_baseline(power)
paired = pair_vibration_with_power(vib, power)

print(f"power readings inside a job:     {power['job_id'].notna().sum():,} of {len(power):,}"
      f" ({100 * power['job_id'].notna().mean():.1f}%)")
print(f"vibration readings inside a job: {vib['job_id'].notna().sum():,} of {len(vib):,}"
      f" ({100 * vib['job_id'].notna().mean():.1f}%)")
print(f"jobs with at least one power reading: {power['job_id'].nunique()} of {len(log)}")
print(f"vibration rows with no power within 1 s: {int(paired['power_kw'].isna().sum()):,}")
print(f"\nidle baseline (kW):\n{idle_baseline.round(3).to_string()}")

power readings inside a job:     39,350 of 86,377 (45.6%)
vibration readings inside a job: 75,689 of 168,961 (44.8%)
jobs with at least one power reading: 94 of 94
vibration rows with no power within 1 s: 199

idle baseline (kW):
machine_id
CNC-07    0.290
CNC-09    0.302
CNC-11    0.321


## 5. Features, and the shape of the vibration/power relationship

The key derived quantity is **vibration per net kW** — the raw physical ratio behind the
whole exercise.

The scatter below shows vibration rising with power *multiplicatively*: spread widens
with the level. So the model is fitted in **log space**, where `log(rms) ~ log(net power)`
makes the error additive and roughly homoscedastic. Without it, one sigma cannot serve
both a 3 kW aluminium cut and a 15 kW titanium cut — and a residual in logs reads
directly as *"x% more vibration than a healthy tool would give"*.

Readings below `CUT_FLOOR_KW` are dropped: inside a job window a machine still spends
time not really cutting, where the ratio explodes on a near-zero denominator.

In [6]:
def build_reading_table(paired_df, jobs_df, idle):
    '''One row per in-job vibration reading, joined to power and job context.'''
    df = paired_df[paired_df["job_id"].notna() & paired_df["power_kw"].notna()].copy()
    df = df.merge(jobs_df[["job_id", "part_type", "operator", "quantity", "duration_s"]],
                  on="job_id", how="left", validate="many_to_one")

    df["idle_baseline_kw"] = df["machine_id"].map(idle).astype(float)
    df["net_power_kw"] = (df["power_kw"] - df["idle_baseline_kw"]).clip(lower=0.0)
    df["vib_per_kw"] = df["vibration_rms_g"] / df["net_power_kw"].clip(lower=0.05)
    df["peak_to_rms"] = df["vibration_peak_g"] / df["vibration_rms_g"].clip(lower=1e-6)

    df["log_net_power_kw"] = np.log(df["net_power_kw"].clip(lower=0.05))
    df["log_vibration_rms_g"] = np.log(df["vibration_rms_g"].clip(lower=1e-4))
    df["log_vib_per_kw"] = df["log_vibration_rms_g"] - df["log_net_power_kw"]
    return df.reset_index(drop=True)

In [7]:
readings = build_reading_table(paired, log, idle_baseline)
cutting = readings[readings["net_power_kw"] >= CUT_FLOOR_KW].reset_index(drop=True)

print(f"in-job readings {len(readings):,} -> cutting readings {len(cutting):,} "
      f"across {cutting['job_id'].nunique()} jobs")
print(f"machines: {sorted(cutting['machine_id'].unique())}")
print(f"part types: {sorted(cutting['part_type'].dropna().unique())}")

# Quantify the heteroscedasticity that justifies the log target.
for label, col in [("additive", "vibration_rms_g"), ("log", "log_vibration_rms_g")]:
    sextile = pd.qcut(cutting["net_power_kw"], 6, labels=False, duplicates="drop")
    spread = cutting.groupby(sextile)[col].std()
    print(f"{label:>9} target: residual-proxy spread across power sextiles "
          f"varies {spread.max() / spread.min():.2f}x")

in-job readings 75,689 -> cutting readings 73,379 across 91 jobs
machines: ['CNC-07', 'CNC-09', 'CNC-11']
part types: ['AL-bracket', 'AL-panel', 'ST-gear', 'ST-housing', 'TI-fitting']
 additive target: residual-proxy spread across power sextiles varies 2.31x
      log target: residual-proxy spread across power sextiles varies 1.42x
